In [ ]:
# Libraries 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time 
import re 
import warnings
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler


In [ ]:
# mortality rates 
africa = pd.read_csv("/kaggle/input/data-asssignment/Child mortality rates_Africa.csv")
americas = pd.read_csv('/kaggle/input/data-asssignment/Child mortality rates_Americas.csv')
mediterranean = pd.read_csv('/kaggle/input/data-asssignment/Child mortality rates_Eastern_Mediterranean.csv')
europe = pd.read_csv('/kaggle/input/data-asssignment/Child mortality rates_Europe.csv')
south_east = pd.read_csv('/kaggle/input/data-asssignment/Child mortality rates_South_East_Asia.csv')
western_pacific = pd.read_csv('/kaggle/input/data-asssignment/Child mortality rates_Western_Pacific.csv')

# infant nutrition 
nutrition_data = pd.read_csv('/kaggle/input/data-asssignment/Infant nutrition data by country.csv')

In [ ]:
# combine mortality datasets

mortality_data = pd.concat([africa, americas, mediterranean, europe, south_east, western_pacific], ignore_index=True)

mortality_data.tail()

In [ ]:
# remove unnecessary columns and rename columns for better clarity 

mortality_data.columns = mortality_data.iloc[0]
mortality_data = mortality_data.drop(0).reset_index(drop=True)

mortality_data.columns = ['Country', 'Year', 'Mortality Both Sexes', 'Mortality Male', 'Mortality Female',
                          'Death Both Sexes', 'Death Male', 'Death Female']

In [ ]:
# rename 'country, territory, regions' column in nutrition data the same as it is in mortality_data

nutrition_data.rename(columns={nutrition_data.columns[0]: 'Country'}, inplace=True)

In [ ]:
# Clean nutrition_data, convert year ranges like 2006-2007 to their average
def process_year(year_str):
    if isinstance(year_str, str) and '-' in year_str: 
        years = year_str.split('-')
        return (int(years[0]) + int(years[1])) // 2
    elif isinstance(year_str, str): 
        return int(year_str)
    else: 
        return year_str

nutrition_data['Year'] = nutrition_data['Year'].apply(process_year)
mortality_data['Year'] = pd.to_numeric(mortality_data['Year'], errors='coerce')

In [ ]:
# Randomise the initial solution by setting random years for mortality_data based on years in nutrition_data
def randomise_initial_solution(nutrition_data, mortality_data):
    randomised_data = mortality_data.copy()
    possible_years = nutrition_data['Year'].unique()
    # Assign a closer starting year by using a smaller range of years around the mean year
    mean_year = int(np.mean(possible_years))
    randomised_data['Year'] = [
        random.choice([year for year in possible_years if abs(year - mean_year) <= 5])
        for _ in range(len(randomised_data))
    ]
    return randomised_data

# Calculate the total cost between nutrition_data and mortality_data
def calculate_cost(nutrition_data, mortality_data):
    total_cost = 0
    for _, row in nutrition_data.iterrows():
        country = row['Country']
        year = row['Year']
        
        # Filter mortality data for the country
        matching_rows = mortality_data[mortality_data['Country'] == country]
        
        # Calculate cost as distance to closest year in mortality_data
        if not matching_rows.empty:
            mortality_years = matching_rows['Year'].values
            cost = np.min(np.abs(mortality_years - year))
            cost = min(cost, 5)  # Cap cost to 5
        else:
            cost = 10  # Penalty for no matching country
        
        total_cost += cost

    return total_cost

# Generate a neighbor solution by making small changes to current solution
def generate_neighbor(individual, nutrition_data):
    neighbor = individual.copy()
    mutation_indices = random.sample(range(len(neighbor)), k=max(1, int(len(neighbor) * 0.1)))  # Mutate 10% of rows
    for index in mutation_indices:
        country = neighbor.iloc[index]['Country']
        # Limit possible years to those for the selected country in nutrition_data
        possible_years = nutrition_data[nutrition_data['Country'] == country]['Year'].unique()
        if possible_years.size > 0:
            neighbor.at[index, 'Year'] = random.choice(possible_years)
    return neighbor

# Simulated Annealing Algorithm
def simulated_annealing(nutrition_data, mortality_data, initial_temp, cooling_rate, max_iters):
    current_solution = randomise_initial_solution(nutrition_data, mortality_data)
    current_cost = calculate_cost(nutrition_data, current_solution)
    best_solution = current_solution.copy()
    best_cost = current_cost

    temp = initial_temp
    costs = [current_cost]

    for i in range(max_iters):
        # Generate neighbor and calculate new cost
        new_solution = generate_neighbor(current_solution, nutrition_data)
        new_cost = calculate_cost(nutrition_data, new_solution)

        # Acceptance criteria
        if new_cost < current_cost or random.uniform(0, 1) < np.exp((current_cost - new_cost) / temp):
            current_solution = new_solution
            current_cost = new_cost
            
            # Update best solution if new solution is better
            if new_cost < best_cost:
                best_solution = new_solution
                best_cost = new_cost
        
        # Track cost over iterations and decrease temperature
        costs.append(current_cost)
        temp *= cooling_rate  

    return best_solution, best_cost, costs

# Parameters
initial_temp = 100
cooling_rate = 0.99
max_iters = 20


In [ ]:
# Run the Simulated Annealing optimization

start = time.time()
best_solution, best_cost, costs = simulated_annealing(nutrition_data, mortality_data, initial_temp, cooling_rate, max_iters)

end = time.time()

total_time = end - start


print("Best Cost:", best_cost)
print("Execution time :", total_time)

# Plot the costs
plt.plot(costs)
plt.title('Simulated Annealing Optimization Costs')
plt.xlabel('Iteration')
plt.ylabel('Cost')
plt.grid(True)
plt.show()



In [ ]:
# Merge alligned datasets

final_data = pd.merge( best_solution, nutrition_data,  on= ['Country', 'Year'], how = 'inner')


In [ ]:
final_data.head()

In [ ]:
# Inner join mortality_data and nutrition_data

inner_join = pd.merge(nutrition_data, mortality_data, on= ['Year', 'Country'], how = 'inner')

In [ ]:
# 10% Sample data from SA algorithm and Inner join 

sa_sample = final_data.sample(frac = 0.10)
inner_sample = inner_join.sample(frac = 0.10)


In [ ]:

# Function to calculate cost for a sample
def calculate_sample_cost(sample, nutrition_data):
    total_cost = 0
    for _, row in sample.iterrows():
        country = row['Country']
        year = row['Year']
        matching_rows = nutrition_data[nutrition_data['Country'] == country]
        if not matching_rows.empty:
            nutrition_years = matching_rows['Year'].values
            cost = np.min(np.abs(nutrition_years - year))
            if cost > 5:
                cost = 5  # Limit the cost to a maximum of 5
        else:
            cost = 10  # Penalty for no matching country
        total_cost += cost
    return total_cost

# Calculate costs for each sample
sa_sample_cost = calculate_sample_cost(sa_sample, nutrition_data)
inner_sample_cost = calculate_sample_cost(inner_sample, nutrition_data)

# Print the costs for comparison
print("SA Sample Cost:", sa_sample_cost)
print("Inner Join Sample Cost:", inner_sample_cost)



In [ ]:
# Clean data by removing strings/ranges like e.g.'[12-23]'

columns_to_clean = [
                   'Mortality Both Sexes', 'Mortality Male', 'Mortality Female',
                   'Death Both Sexes', 'Death Male', 'Death Female', 'Early initiation of breastfeeding (%)',
                   'Infants exclusively breastfed for the first six months of life (%)']


# Ensure 'NaN' string values are converted to actual NaN 

final_data.replace('NaN', np.nan, inplace=True)

# Function to extract the first value 
def extract_first_value(text):
    if pd.isnull(text):  
        return np.nan
    text = re.sub(r'\s', '', str(text))  # Remove all whitespace characters and convert to string
    match = re.search(r'(\d+\.?\d*)', text)  # Find the first number (with optional decimal)
    if match:
        return float(match.group(1))  # Convert it to a float
    else:
        return np.nan  # Return NaN if no number is found


for column in columns_to_clean:
    final_data[column] = final_data[column].apply(extract_first_value)  

final_data.head()  

In [ ]:
# Fill missing values with the median for specified columns 

# Ensure 'NaN' string values are converted to actual NaN values
final_data.replace('NaN', np.nan, inplace=True)

final_data = final_data.replace('NaN', np.nan) 
for col in final_data.columns[2:]: 
    if final_data[col].isna().any(): 
        final_data[col] = final_data[col].astype(float).fillna(final_data[col].astype(float).median())

In [ ]:
final_data.head()

In [ ]:
final_data.to_csv("final_data.csv", index = False)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

# Select features (X) and target (y)
X = final_data[['Early initiation of breastfeeding (%)', 'Infants exclusively breastfed for the first six months of life (%)']]
y = final_data['Mortality Both Sexes']


# Scale the features using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data into training and testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.4, random_state=42)

# Create and train the neural network model with the new architecture
# 2 hidden layers, 1000 neurons per layer, learning rate 0.1, activation function 'relu'
nn_model = MLPRegressor(hidden_layer_sizes=(2000, 2000), 
                        max_iter=1500, 
                        random_state=42, 
                        activation='relu',  
                        learning_rate_init=0.1, 
                        early_stopping=True, 
                        validation_fraction=0.1)

# Train the model
nn_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_nn = nn_model.predict(X_test)

# Evaluate the model using MSE and R²
mse_nn = mean_squared_error(y_test, y_pred_nn)
r2_nn = r2_score(y_test, y_pred_nn)
print(f"Mean Squared Error (Neural Network): {mse_nn}")
print(f"R² (Neural Network): {r2_nn}")

# Plotting the results: Actual vs Predicted
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_nn, color='blue', label='Predicted')  # Scatter plot of actual vs predicted values
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linewidth=2, label='Perfect Prediction Line')  # Perfect prediction line
plt.title('Neural Network - Actual vs Predicted Mortality Rates')
plt.xlabel('Actual Mortality Male')
plt.ylabel('Predicted Mortality Male')
plt.legend()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Select features (X) and target (y)
X = final_data[['Early initiation of breastfeeding (%)', 'Infants exclusively breastfed for the first six months of life (%)']]
y = final_data['Mortality Male']

# Check for missing values and handle them (fill with mean or remove)
X = X.fillna(X.mean())
y = y.fillna(y.mean())

# Scale the features using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data into training and testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Create and train the neural network model with specified architecture
nn_model = MLPRegressor(hidden_layer_sizes=(50, 50), 
                        max_iter=1000, 
                        random_state=42, 
                        activation='relu',  
                        learning_rate_init=0.001, 
                        early_stopping=True, 
                        validation_fraction=0.1,
                        alpha=0.01)  # Adding L2 regularization

# Train the model
nn_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_nn = nn_model.predict(X_test)

# Evaluate the model using MSE and R²
mse_nn = mean_squared_error(y_test, y_pred_nn)
r2_nn = r2_score(y_test, y_pred_nn)
print(f"Mean Squared Error (Neural Network): {mse_nn}")
print(f"R² (Neural Network): {r2_nn}")

# Sort values for cumulative gain plot
sorted_indices = np.argsort(y_pred_nn)[::-1]
y_test_sorted = y_test.values[sorted_indices]
y_pred_sorted = y_pred_nn[sorted_indices]

# Calculate cumulative sums
y_test_cumulative = np.cumsum(y_test_sorted)
y_pred_cumulative = np.cumsum(y_pred_sorted)

# Normalize cumulative values for a comparable range
y_test_cumulative = y_test_cumulative / y_test_cumulative[-1]
y_pred_cumulative = y_pred_cumulative / y_pred_cumulative[-1]

# Plot cumulative gain curve
plt.figure(figsize=(8, 6))
plt.plot(y_test_cumulative, label='Actual Cumulative Mortality', color='green')
plt.plot(y_pred_cumulative, label='Predicted Cumulative Mortality', color='orange')
plt.title('Cumulative Gain Curve - Actual vs Predicted Mortality')
plt.xlabel('Instances (sorted by predicted value)')
plt.ylabel('Cumulative Mortality')
plt.legend()
plt.show()


In [ ]:
# nutrition/mortality data correlation matrix

breastfeeding_indicators = ['Early initiation of breastfeeding (%)', 'Infants exclusively breastfed for the first six months of life (%)']
mortality_rates = ['Mortality Both Sexes', 'Mortality Male', 'Mortality Female', 'Death Both Sexes', 'Death Male', 'Death Female']

subset_df = final_data[[*breastfeeding_indicators, *mortality_rates]]

# Calculate the correlation matrix
correlation_matrix = subset_df.corr()

# Extract the desired portion of the correlation matrix
correlation = correlation_matrix.loc[breastfeeding_indicators, mortality_rates].round(2)


In [ ]:
# Create table of correlation matrix


# Apply styling to the correlation matrix
styled_correlation = correlation.style.background_gradient(cmap='coolwarm')

# Display the styled table
display(styled_correlation)